# moonshot_compounding — MI screen: CALENDAR-FIXED re-run

**Purpose.** Same pipeline as `moonshot_mi_paper_rerun.ipynb`, but on a corrected
trading calendar. Review found that the panel's date index contains exchange-closed
dates on which most stocks are blank; month-ends or forward-window endpoints landing on
them silently dropped ~11 of 144 months. Cell C2 audits every date's coverage, writes the
audit, and removes low-coverage dates BEFORE any positional shift. Stage 1's comparison
with the 2026-08-10 results is now informational: differences are expected. Characterization only, on the
**2010–2021 dev period**; the 2022+ hold-out is never read (guarded in `mi_screen`,
re-audited here).

**What this run does**
1. **Reproduces** the 2026-08-10 screen exactly (same functions, same RNG stream) while
   capturing the per-date MI values the original discarded. Checked against the saved CSV.
2. **95% moving-block-bootstrap CIs** (block ≥ horizon in months). The originals were 90%
   and iid over months, too narrow for 3m/6m/1y where forward windows overlap.
3. **Direction vs magnitude** for every cell: rank IC, MI with the *size* of the forward
   move `|fwd_rank − 0.5|`, and the share of MI that is about size.
4. **Conditional MI** (corrected metric) reproduced with per-date capture → 95% MBB CIs.
5. **Redundancy cross-check**: Spearman for every feature pair + the MI a Gaussian copula
   with that Spearman implies; plus the redundancy MI matrix under the current code.
6. **KSG calibration** against known Gaussian MI at n≈480.

### Pre-registered classification rule (fixed before this run)
For each (feature, horizon), with 95% MBB CIs:
- `no_signal` — MI_dir CI does not exclude 0
- `magnitude-led` — MI_mag / MI_dir ≥ **0.5** and MI_mag CI excludes 0
- `direction-led` — share < 0.5 and rank-IC CI excludes 0
- `mixed/unclear` — otherwise

Two new analyses (rank IC; magnitude MI) count toward the running test tally.

**Effective-settings note.** The original drivers set `N_SHUFFLE=15`, `N_BOOT=500` after
import, but the functions bound their defaults at import, so the screen actually ran with
**40 shuffles / 300 bootstrap draws**. Cell A prints what really runs; the paper quotes that.

**Run order:** A → B (gates) → C → stages 1–6 → manifest. Each stage saves its output and
**skips if the output already exists**, so a Colab disconnect costs only the stage in
progress. Expect ~60 min end to end.


## A — mount, paths, modules

In [ ]:
from pathlib import Path
import sys, json, time, platform, importlib, subprocess, datetime as dt
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("mount note:", e)

BASE = Path("/content/drive/MyDrive/moonshot_compounding")
assert BASE.exists(), f"BASE not found: {BASE}"
n_parquet = len(list((BASE/"features").glob("*.parquet")))
assert n_parquet >= 2, "Too few parquet files visible -> STALE MOUNT. Runtime > Disconnect and delete runtime, remount."

MODULE_DIR = BASE / "code" / "mi_screen"
for f in ["mi_screen.py", "conditional_mi.py", "synthetic_selftest.py",
          "conditional_mi_selftest.py", "mi_paper_rerun.py"]:
    assert (MODULE_DIR/f).exists(), f"{f} missing from {MODULE_DIR}"

subprocess.run([sys.executable, "-m", "pip", "install", "npeet-plus", "-q"], check=False)
sys.path.insert(0, str(MODULE_DIR))

# Import order matters: conditional_mi binds its defaults from mi_screen AT IMPORT,
# exactly as in the original notebooks. Fresh imports, no attribute changes before this.
import mi_screen as M, conditional_mi as C, synthetic_selftest as S
import conditional_mi_selftest as CS, mi_paper_rerun as R
for mod in (M, C, S, CS, R):
    importlib.reload(mod)

TODAY = dt.date.today().isoformat()
RUN_DIR = BASE / "hypotheses" / "mi_screen" / f"paper_rerun_calfix_{TODAY}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
SAVED_DIR = BASE / "hypotheses" / "mi_screen" / "run_2026-08-10"
out = lambda name: RUN_DIR / f"{name}_{TODAY}"

SETTINGS = R.effective_settings()
for k, v in SETTINGS.items():
    print(f"  {k}: {v}")
N_SHUFFLE_EFFECTIVE = SETTINGS["per_date_mi_excess.n_shuffle(default bound at import)"]
STATUS, T0 = {}, time.time()
print("run dir:", RUN_DIR)


## B — self-test gates (must all pass before real data)
1. Original screen self-test (planted signal, noise, leakage, PIT, hold-out, NaN-roll).
2. Corrected conditional-MI metric gate (known-redundant vs known-additive), as in the
   2026-09 correction notebook.
3. New machinery: moving-block bootstrap, direction-vs-magnitude classification on planted
   features, copula identity, KSG calibration.

In [ ]:
S.main()
print("\n>>> GATE 1 PASSED\n")

panel_s, A_, B_, I_ = CS.build_synthetic()
M.FEATURES["redundant_A"] = lambda p, w=A_: w
M.FEATURES["redundant_B"] = lambda p, w=B_: w
M.FEATURES["independent"] = lambda p, w=I_: w
M.N_SHUFFLE = 8
try:
    r_red = C.corrected_pair(panel_s, "redundant_A", "redundant_B", "1m", n_noise_draws=5)
    r_add = C.corrected_pair(panel_s, "independent", "redundant_A", "1m", n_noise_draws=5)
    assert not r_red["additive"], "GATE 2 FAIL: redundant pair flagged additive"
    assert r_add["additive"], "GATE 2 FAIL: additive pair not flagged"
finally:
    for k in ["redundant_A", "redundant_B", "independent"]:
        M.FEATURES.pop(k, None)
print("\n>>> GATE 2 PASSED\n")

R.selftest()
print("\n>>> GATE 3 PASSED — safe to run on real data.")


## C — load the real panel + the saved 2026-08-10 results

In [ ]:
def newest(folder, stem):
    fs = sorted((BASE/folder).glob(f"{stem}_*.parquet"))
    assert fs, f"no {stem}_*.parquet in {folder}"
    return fs[-1]

dp_path = newest("features", "feature_panel_with_pit_v2")
am_path = newest("features", "amihud_monthly_v2")
print("daily :", dp_path.name, "\namihud:", am_path.name)
daily_raw = pd.read_parquet(dp_path)
amihud_raw = pd.read_parquet(am_path)

# ---- C2: calendar audit -------------------------------------------------------
# Coverage is measured over ALL tickers (not index members: membership is unknown
# before the first snapshot in mid-2010, and those dates are real sessions whose
# prices momentum needs). For each date: number of tickers with a Close, relative to
# the median of the surrounding 21 dates (centred). The universe grows over time, so a
# local median, not a global one. A closed date carried by a handful of rows, or a
# thin special session, falls far below its neighbours.
dr = daily_raw.copy()
dr["Date"] = pd.to_datetime(dr["Date"]).astype("datetime64[ns]")
all_dates = pd.DatetimeIndex(sorted(dr["Date"].unique()))
n_close = dr.groupby("Date")["Close"].apply(lambda s: s.notna().sum()).reindex(all_dates).fillna(0)
cov = n_close.rename("n_with_close").to_frame()
cov["local_median"] = n_close.rolling(21, center=True, min_periods=5).median()
cov["coverage"] = cov["n_with_close"] / cov["local_median"]
cov["weekday"] = cov.index.day_name()
COV_MIN = 0.5                            # pre-set rule: < half the local median
cov["closed_or_thin"] = cov["coverage"] < COV_MIN
cov.to_csv(out("calendar_audit").with_suffix(".csv"))
bad = cov.index[cov["closed_or_thin"]]
med = cov["coverage"].median()
print(f"median relative coverage {med:.3f}; threshold {COV_MIN:.3f}")
print(f"dates flagged closed/thin: {len(bad)} of {len(cov)}")
print(cov.loc[bad, ["coverage", "n_with_close", "local_median", "weekday"]].to_string())
watch = ["2011-08-31","2014-08-29","2015-02-28","2016-10-31","2018-03-29","2018-03-30",
         "2019-03-29","2020-11-30","2014-04-24","2019-10-21","2021-03-29"]
w = [pd.Timestamp(x) for x in watch if pd.Timestamp(x) in cov.index]
print("\nwatch-list dates:"); print(cov.loc[w, ["coverage", "n_with_close", "weekday"]].to_string())

# Remove flagged dates from the panel BEFORE build_panel, so every positional shift
# (features, forward returns, month-end grid) runs on real sessions only.
# NOTE: realized_vol_60 and daily/overnight returns are precomputed in the panel and
# are NOT rebuilt here; that is documented as a limitation.
daily_fixed = daily_raw[~pd.to_datetime(daily_raw["Date"]).astype("datetime64[ns]").isin(bad)]
print(f"\nrows removed: {len(daily_raw) - len(daily_fixed):,}")
panel = M.build_panel(daily_fixed, amihud_raw)
print("panel:", len(panel.dates), "dates x", len(panel.tickers), "tickers")

reb = M.month_end_rebalance_dates(panel)
vf = [t for t in reb if panel.date_status.get(t, "ok") == "vendor_failure"]
assert len(vf) == 0, f"rebalance dates on vendor_failure: {vf}"
print(f"train rebalances: {len(reb)} ({reb.min().date()} -> {reb.max().date()})")
assert reb.max() <= M.TRAIN_END

saved_land = pd.read_csv(SAVED_DIR / "mi_landscape_2026-08-10.csv")
saved_cmi = pd.read_csv(SAVED_DIR / "conditional_mi" / "conditional_mi_corrected_2026-08-10.csv")
print("saved landscape rows:", len(saved_land), "| saved CMI rows:", len(saved_cmi))


## Stage 1 — reproduce the screen, capturing per-date MI
Identical call to the original `run_screen`; the recorder only copies outputs. The
reproduced point estimates are compared with the saved CSV. A mismatch is reported loudly
(it would mean a library-version difference), not silently accepted.

In [ ]:
p1 = out("per_date_dir").with_suffix(".parquet")
if p1.exists():
    per_date_dir = pd.read_parquet(p1); STATUS["stage1"] = "loaded"
    print("loaded", p1.name)
else:
    t = time.time()
    res, per_date_dir = R.reproduce_screen(panel, verbose=True)
    per_date_dir.to_parquet(p1)
    chk = R.compare_to_saved(res, saved_land)
    chk.to_csv(out("reproduction_check").with_suffix(".csv"), index=False)
    print(chk.round(5).to_string(index=False))
    n_ok = int(chk["match"].sum())
    STATUS["stage1"] = f"ran {(time.time()-t)/60:.1f} min; reproduced {n_ok}/{len(chk)} cells within 0.002"
    if n_ok < len(chk):
        print("\n(informational) not all cells match the old calendar - expected after the fix.")
print(STATUS["stage1"])
print("hold-out audit: rows checked =", R.audit_holdout(panel, per_date_dir), "-> no forward window past", M.TRAIN_END.date())


## Stage 2 — direction (rank IC) and magnitude MI
Same cross-sections as Stage 1 (asserted), same estimator and shuffle count.

In [ ]:
p2 = out("per_date_full").with_suffix(".parquet")
if p2.exists():
    per_date = pd.read_parquet(p2); STATUS["stage2"] = "loaded"
else:
    t = time.time()
    per_date = R.direction_magnitude_pass(panel, per_date_dir, n_shuffle=N_SHUFFLE_EFFECTIVE)
    per_date.to_parquet(p2)
    STATUS["stage2"] = f"ran {(time.time()-t)/60:.1f} min"
print(STATUS["stage2"], "| rows:", len(per_date))


## Stage 3 — 95% moving-block CIs + pre-registered classification

In [ ]:
land = R.summarise_cells(per_date, saved_land)
land.to_csv(out("landscape_v2").with_suffix(".csv"), index=False)
show = ["feature","horizon","n_dates","mi_dir","mi_dir_lo","mi_dir_hi","saved_ci90_lo",
        "ic_dir","ic_dir_lo","ic_dir_hi","mi_mag","mag_share","class"]
print(land[show].round(4).to_string(index=False))
print("\nclass counts:\n", land["class"].value_counts().to_string())


## Stage 4 — corrected conditional MI, reproduced with per-date capture
Mirrors the 2026-09 correction notebook: `N_SHUFFLE` attribute 15 (sets the floor's
shuffle count to 7, as originally run), 10 noise draws, the four pre-registered pairs.

In [ ]:
p4 = out("cmi_v2").with_suffix(".csv")
if p4.exists():
    cmi = pd.read_csv(p4); STATUS["stage4"] = "loaded"
else:
    t = time.time()
    M.N_SHUFFLE = 15
    pairs = [("momentum_12_1","rvol_60"), ("rvol_60","momentum_12_1"),
             ("reversal_1m","momentum_12_1"), ("amihud_illiq","log_turnover_21")]
    cmi, cmi_long = R.reproduce_corrected_cmi(panel, C, pairs, "1m", n_noise_draws=10)
    cmi_long.to_parquet(out("cmi_per_date").with_suffix(".parquet"))
    cmi.to_csv(p4, index=False)
    s = saved_cmi.set_index("pair")["excess_above_floor"]
    cmi["saved_excess"] = cmi["pair"].map(s)
    cmi.to_csv(p4, index=False)
    STATUS["stage4"] = f"ran {(time.time()-t)/60:.1f} min"
print(STATUS["stage4"])
print(cmi[["pair","n_dates","excess_above_floor","saved_excess","ci95_mbb_lo","ci95_mbb_hi","verdict"]].round(4).to_string(index=False))


## Stage 5 — redundancy cross-check
Spearman per pair + copula-implied MI (null-bias corrected, comparable to KSG MI_excess).
If KSG MI ≈ implied, the dependence is essentially monotone and rank correlation already
sees it; KSG ≫ implied means genuinely non-monotone structure. Then the redundancy MI matrix
under the current code (~15 min; set `RUN_REDUNDANCY_MI = False` to skip).

In [ ]:
RUN_REDUNDANCY_MI = True

p5a = out("spearman_pairs").with_suffix(".csv")
if p5a.exists():
    sp = pd.read_csv(p5a)
else:
    sp = R.spearman_matrix(panel); sp.to_csv(p5a, index=False)
print(sp.round(4).to_string(index=False))

p5b = out("redundancy_mi").with_suffix(".csv")
if RUN_REDUNDANCY_MI and not p5b.exists():
    t = time.time()
    red = M.feature_redundancy_matrix(panel)
    red.to_csv(p5b)
    json.dump({k: [list(map(str, x)) for x in v] for k, v in red.attrs.items()},
              open(out("redundancy_mi_attrs").with_suffix(".json"), "w"), indent=2)
    STATUS["stage5"] = f"ran {(time.time()-t)/60:.1f} min"
elif p5b.exists():
    red = pd.read_csv(p5b, index_col=0); STATUS["stage5"] = "loaded"
else:
    red = None; STATUS["stage5"] = "skipped"
if red is not None:
    print(red.round(4).to_string())
    ksg = [red.loc[a, b] for a, b in zip(sp.feature_a, sp.feature_b)]
    sp["ksg_mi_excess"] = ksg
    sp["ksg_over_implied"] = sp["ksg_mi_excess"] / sp["copula_implied_mi_excess"]
    sp.to_csv(p5a, index=False)
    print("\n", sp[["feature_a","feature_b","spearman","copula_implied_mi_excess","ksg_mi_excess","ksg_over_implied"]].round(4).to_string(index=False))


## Stage 6 — KSG calibration against known Gaussian MI (synthetic, ~2 min)

In [ ]:
p6 = out("ksg_calibration").with_suffix(".csv")
if p6.exists():
    cal = pd.read_csv(p6)
else:
    cal = R.ksg_calibration(n=480, reps=30); cal.to_csv(p6, index=False)
print(cal.round(4).to_string(index=False))


## Manifest

In [ ]:
from importlib.metadata import version
import sklearn
manifest = {
    "run_date": TODAY, "run_dir": str(RUN_DIR),
    "inputs": {"daily_panel": dp_path.name, "amihud": am_path.name,
               "saved_reference": str(SAVED_DIR)},
    "effective_settings": {k: str(v) for k, v in SETTINGS.items()},
    "versions": {"python": platform.python_version(), "numpy": np.__version__,
                 "pandas": pd.__version__, "sklearn": sklearn.__version__,
                 "npeet_plus": version("npeet-plus")},
    "stages": STATUS, "total_minutes": round((time.time() - T0) / 60, 1),
    "holdout": f"all forward windows <= {M.TRAIN_END.date()} (audited in stage 1)",
}
json.dump(manifest, open(out("manifest").with_suffix(".json"), "w"), indent=2)
print(json.dumps(manifest, indent=2))
print("\nFiles:"); [print("  ", p.name) for p in sorted(RUN_DIR.iterdir())]


---
**Done.** Tell Claude the run finished; the outputs are read straight from
`hypotheses/mi_screen/paper_rerun_<date>/` on Drive.